### Tsetlin Machine feature net regressors

This notebook trains separate, frozen auxiliary (aux) TM models and feeds what they produce into a fresh main TM as extra Boolean inputs. The main training set has **4 CYP isoforms** with 4 pIC50 columns. 

**Pipeline**

1. **Canonicalise SMILES** in every data file and report overlaps between aux, main and test sets.
2. **Train auxiliary TMs once** (never retrained, never see main labels):
   - **Aux A**: either *wide* (one TM per lab value × CYP isoform) or *long* (one pooled TM per lab value, with the isoform one-hot encoded as input bits). Set `AUX_A_FORMAT`.
   - **Aux B**: one TM per isoform column.
   - **Aux C**: one TM per isoform column (3 columns, each mapped to one of the main isoforms with `AUX_C_COL_MAP`; the remaining main isoform simply has no aux C features).
3. **Turn each aux TM into features** for every main and test molecule:
   - **Clause outputs**: whether each of the top-N informative aux clauses fires (0/1).
   - **Predictions**: the aux TM's predicted values are thermometer-encoded into bits.
   - **Cutoff bits** (for aux A, optional): one bit per block for "predicted lab value **below** its known cutoff", the range of interest (`AUX_A_THRESHOLDS`).
4. **Append** those bits to the 2048 ECFP bits and train one main `TMRegressor` per isoform on the augmented input.
5. **Predict test** by running the same pipeline: ECFP → frozen aux TMs → append → main TM.

In [ ]:
import os
import sys
import pickle
import hashlib
from joblib import Parallel, delayed
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import optuna
from scipy.special import expit

from tm_qsar_benchmark.descriptors import mol_from_smiles, gen_ecfp_arr 
# Public vanilla regressor. NOTE: TMRegressor is SINGLE-OUTPUT, so multi-target =
# one model per target column (for the aux TMs and the main TMs alike).
from tmu.models.regression.vanilla_regressor import TMRegressor
from sklearn.metrics import (
    precision_recall_curve, auc, precision_score, recall_score,
    roc_curve, roc_auc_score, mean_squared_error, mean_absolute_error, r2_score,
)
from sklearn.model_selection import train_test_split, GroupKFold
from collections import defaultdict
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold

from importlib.metadata import version as _version, PackageNotFoundError


def _v(pkg, module=None):
    # Installed version of a package (pip or conda), without printing any paths.
    try:
        return _version(pkg)
    except PackageNotFoundError:
        try:
            return getattr(__import__(module or pkg), "__version__", "installed")
        except ImportError:
            return "not found"


# Versions only (no file paths), for reproducibility
print(f"Python {sys.version.split()[0]} | " + " | ".join(
    f"{p} {_v(p, m)}" for p, m in [("tmu", None), ("rdkit", None), ("numpy", None), ("pandas", None),
                                    ("scikit-learn", "sklearn"), ("joblib", None), ("optuna", None)]))

In [ ]:
def prc_auc_score(Y, Y_pred):
    precision, recall, _ = precision_recall_curve(Y, Y_pred)
    return auc(recall, precision)


def ppv_npv_score(Y, Y_pred):
    fpr, tpr, proba = roc_curve(Y, Y_pred)
    optimal_proba_cutoff = sorted(
        list(zip(np.abs(tpr - fpr), proba)), key=lambda i: i[0], reverse=True
    )[0][1]
    hard_Y_pred = [1 if p > optimal_proba_cutoff else 0 for p in Y_pred]
    return (
        precision_score(Y, hard_Y_pred, pos_label=1),
        precision_score(Y, hard_Y_pred, pos_label=0),
    )


def select_best_threshold(y_true, y_scores, min_precision=0.75):
    """recall@precision: highest-recall threshold meeting a minimum precision."""
    precision, recall, thresholds = precision_recall_curve(y_true, y_scores)
    eligible = np.flatnonzero(precision[:-1] >= min_precision)
    if eligible.size == 0:
        return float(np.mean(y_scores))
    best_idx = eligible[np.argmax(recall[:-1][eligible])]
    return thresholds[best_idx]


def scale01(y):
    """Min-max scale a 1-D target to [0, 1]; return (scaled, (lo, hi)).

    Each TM here has its own targets, so this is only for tidy, comparable ranges
    (and for the per-isoform scaling option of the long aux-A model).
    """
    y = np.asarray(y, dtype=np.float64)
    lo, hi = float(np.nanmin(y)), float(np.nanmax(y))
    if hi == lo:
        hi = lo + 1.0
    return ((y - lo) / (hi - lo)).astype(np.float32), (lo, hi)


def unscale01(y, bounds):
    lo, hi = bounds
    return np.asarray(y, dtype=np.float64) * (hi - lo) + lo


def canonical_smiles(smiles):
    """RDKit canonical SMILES (isomeric), or None if the SMILES cannot be parsed.

    Used as the molecule KEY everywhere: overlap/leakage checks, de-duplication,
    fingerprint caching and grouping, so that the same molecule written two
    different ways in two files is recognised as one molecule.
    """
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        return None
    return Chem.MolToSmiles(mol, isomericSmiles=True)


def murcko_scaffold(smiles):
    """Bemis-Murcko scaffold SMILES. Acyclic molecules share the empty scaffold ""."""
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        return f"__unparsed__{smiles}"   # unparseable SMILES -> its own group
    return MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False)


def scaffold_split(idx, scaffolds, val_fraction):
    """Split row indices so no scaffold appears in both train and validation.

    Scaffold groups are sorted largest first and fill the training set; groups
    that no longer fit go to validation. Validation therefore holds the rarer
    scaffolds, which is the stricter, more realistic QSAR test. Deterministic.
    """
    groups = defaultdict(list)
    for i in idx:
        groups[scaffolds[i]].append(i)
    ordered = sorted(groups.values(), key=lambda g: (len(g), -g[0]), reverse=True)
    n_train_target = len(idx) - int(round(val_fraction * len(idx)))
    tr, va = [], []
    for g in ordered:
        if len(tr) + len(g) <= n_train_target or not tr:
            tr.extend(g)
        else:
            va.extend(g)
    return np.array(sorted(tr), dtype=int), np.array(sorted(va), dtype=int)


def kennard_stone_split(idx, X, val_fraction):
    """Kennard-Stone split on Tanimoto distance between ECFP bit vectors.

    The TRAINING set is chosen to span chemical space: start with the two most distant
    molecules, then repeatedly add the molecule farthest from everything already chosen.
    The remaining molecules form validation set, so validation lies *inside* the training
    coverage (an interpolation test, usually easier than a scaffold split). Works on
    individual molecules, so close analogues can fall on both sides. Deterministic.
    """
    idx = np.asarray(idx)
    n = len(idx)
    n_train = max(2, n - int(round(val_fraction * n)))
    if n_train >= n:
        return np.sort(idx), np.array([], dtype=int)
    Xs = np.asarray(X[idx], dtype=np.float32)
    counts = Xs.sum(axis=1)
    inter = Xs @ Xs.T
    union = counts[:, None] + counts[None, :] - inter
    with np.errstate(invalid="ignore", divide="ignore"):
        dist = (1.0 - np.where(union > 0, inter / union, 1.0)).astype(np.float32)
    del inter, union
    i, j = np.unravel_index(int(np.argmax(dist)), dist.shape)
    selected = np.zeros(n, dtype=bool)
    selected[[i, j]] = True
    min_d = np.minimum(dist[i], dist[j])
    for _ in range(n_train - 2):
        k = int(np.argmax(np.where(selected, -1.0, min_d)))
        selected[k] = True
        min_d = np.minimum(min_d, dist[k])
    return np.sort(idx[selected]), np.sort(idx[~selected])


def main_split(idx):
    """The single train/validation split of the training cell, following SPLIT_METHOD.

    Shared by the training cell and the Optuna cell, so tuning always stays inside the
    training molecules the rest of the notebook uses.
    """
    if VAL_FRACTION > 0 and len(idx) >= 10:
        if SPLIT_METHOD == "scaffold":
            return scaffold_split(idx, main_scaffolds, VAL_FRACTION)
        if SPLIT_METHOD == "kennard_stone":
            return kennard_stone_split(idx, X_main, VAL_FRACTION)   # on ECFP bits only
        return train_test_split(idx, test_size=VAL_FRACTION, random_state=VAL_RANDOM_STATE)
    return idx, np.array([], dtype=int)

In [ ]:
# ---- Configuration --------------------------------------------------------
# ECFP fingerprint settings (identical for every dataset -> identical booleanization)
FP_RADIUS = 3
FP_SIZE = 2048

# Activity cut-off (original pIC50 units) for deriving a per-isoform binary label (optional; for classification reference)
ACTIVE_CUTOFF = 4.5
MIN_PRECISION = 0.75

RANDOM_SEED = 42

# Auxiliary toggles (the ablation cell also compares main only and combinations of A, B, C)
USE_AUX_A = True
USE_AUX_B = True
USE_AUX_C = True

# ---- Aux A modelling format --------------------------------------------------
# Both options read the SAME long aux A file; this only changes how it is modelled.
# "long": ONE pooled TM per lab value; input = ECFP + 4 isoform one-hot bits.
#         At feature time each molecule is run through it once per isoform setting.
# "wide": pivot to SMILES x isoform; ONE TM per lab value x isoform (up to 8 TMs).
# See the "Aux A: long vs wide" notes below for the trade-offs.
AUX_A_FORMAT = "long"
AUX_A_SCALE_PER_ISOFORM = True   # long only: min-max each isoform's values separately
AUX_A_BALANCE_ISOFORMS = True    # long only: oversample rarer isoforms to the largest one

# ---- What each aux TM contributes to the main TM ------------------------------
# "clauses":     top-N informative clause outputs (0/1) from each aux TM
# "predictions": the aux TM's predicted value, thermometer-encoded
# "both":        both of the above
AUX_FEATURE_MODE = "predictions"
TOP_CLAUSES_PER_MODEL = 64   # clause features kept per aux TM (per isoform setting)
N_THERMO_BITS = 10           # thermometer bits per aux prediction (quantile thresholds)

# Known cutoff per aux-A lab value, in ORIGINAL units (same for every isoform).
# Values BELOW the cutoff are the ones of interest, so each aux-A block gets one extra
# bit: "aux TM predicts this lab value < its cutoff" (1 = below = of interest).
# Added whatever AUX_FEATURE_MODE is; set a value to None (or {} overall) to switch it off.
AUX_A_THRESHOLDS = {
    "log2fc_estimate": -1,   # <-- indicates hits
    "log2fc_fdr": 0.05,   # <-- indicates hits
}

# Which aux features main model j sees:
# "matching": only isoform j aux features
# "all":      aux features for all 4 isoforms (lets one isoform's aux signal help another)
AUX_FEATURE_SCOPE = "all"

# ---- Overlap / leakage handling (see the leakage notes below) -------------------
# "drop_from_aux": remove molecules that are also in MAIN or TEST from aux training
#                  (works with every AUX_FEATURE_MODE).
# "oof":           keep them; their aux PREDICTIONS come from a fold model that did
#                  not see them (out-of-fold). Only valid with AUX_FEATURE_MODE =
#                  "predictions", because clause indices differ between fold models.
# "ignore":        no handling (in-sample aux features; for comparison only).
OVERLAP_HANDLING = "oof"
AUX_OOF_FOLDS = 5

# ---- Aux TM hyperparameters -----------------------------------------------------
AUX_N_CLAUSES = 1600
AUX_T_PER_CLAUSE = 8182 / 1600   # T scales with clauses; same ratio as the main TM
AUX_S = 5.289
AUX_A_LONG_CLAUSE_MULT = 2         # the pooled long model learns 4 isoforms -> more clauses
EPOCHS_AUX_A = 50
EPOCHS_AUX_B = 50
EPOCHS_AUX_C = 50

# Aux TMs train on one CPU core each; independent trainings (for every aux model and with
# out-of-fold handling, each of its fold models) run side by side in many cores.
# Results are identical for any value; only speed and memory change. Each parallel job
# holds its own copy of its training data (aux C: ~1-1.5 GB per job). 1 = one at a time.
N_PARALLEL_JOBS = 8

# ---- Main TM -------------------------------------------------------------------------
EPOCHS_MAIN = 93
# Held-out validation split, carved out of the MAIN data per isoform.
VAL_FRACTION = 0
SPLIT_METHOD = "kennard_stone"   # "scaffold", "kennard_stone" or "random" (single split only;
                                 # CV or CV ablation stay scaffold-grouped unless "random")
VAL_RANDOM_STATE = 42
CV_FOLDS = 0                # 0 = off; e.g. 5 for scaffold-grouped CV (main TM only)

# Main TM hyperparameters (per isoform model)
N_CLAUSES = 3000
T = 12085
S = 6.060

# ---- Train & test column names -------------------------------------------------
# MAIN training set
MAIN_SMILES_COL = "SMILES"
MAIN_ISOFORM_COLS = ["CYP1A2_pIC50_direct_inhibition", "CYP2C9_pIC50_direct_inhibition", "CYP2D6_pIC50_direct_inhibition", "CYP3A4_pIC50_direct_inhibition"]

# AUX A: long, 1 SMILES + 1 isoform column + 2 lab-value target columns (provided by CYP Challenge)
AUX_A_SMILES_COL = "SMILES"
AUX_A_ISOFORM_COL = "enzyme"
AUX_A_VALUE_COLS = ["log2fc_estimate", "log2fc_fdr"]  
AUX_A_ISOFORM_MAP = {
    "CYP3A4": "CYP3A4_pIC50_direct_inhibition",
    "CYP2D6": "CYP2D6_pIC50_direct_inhibition",
    "CYP2C9": "CYP2C9_pIC50_direct_inhibition",
    "CYP1A2": "CYP1A2_pIC50_direct_inhibition",
}

# AUX B: wide, 1 SMILES + 4 regression columns mapping 1:1 to MAIN_ISOFORM_COLS (in order) (provided by CYP Challenge)
AUX_B_SMILES_COL = "SMILES"
AUX_B_TARGET_COLS = ["CYP1A2_EmaxVsPosCtrl_direct_inhibition", "CYP2C9_EmaxVsPosCtrl_direct_inhibition", "CYP2D6_EmaxVsPosCtrl_direct_inhibition", "CYP3A4_EmaxVsPosCtrl_direct_inhibition"]  # <-- placeholder names

# AUX C: wide, 1 SMILES + 3 pIC50 columns, each for one of the main isoforms.
# Keys = aux C column names, values = the MAIN_ISOFORM_COLS entry each matches. (provided by own; open-source data)
AUX_C_SMILES_COL = "SMILES"
AUX_C_COL_MAP = {
    "pred(CYP3A4_pIC50)": "CYP3A4_pIC50_direct_inhibition",
    "pred(CYP2C9_pIC50)": "CYP2C9_pIC50_direct_inhibition",
    "pred(CYP2D6_pIC50)": "CYP2D6_pIC50_direct_inhibition",
}

# TEST set: unlabelled by default (molecule name + SMILES only) -> prediction only.
TEST_SMILES_COL = "SMILES"
TEST_NAME_COL = "Molecule_Name"
TEST_TARGET_COLS = MAIN_ISOFORM_COLS        # only used if present in the test file

# ---- Folders (relative to the notebook's own folder: no full paths needed) --------
DATA_DIR = "../data"        # the five CSVs; use "data" if that folder sits beside the notebook
OUTPUT_DIR = "../outputs"   # cache, predictions, pipeline; created if missing
os.makedirs(OUTPUT_DIR, exist_ok=True)

PREDICTIONS_CSV = os.path.join(OUTPUT_DIR, "test_predictions_final_1_full_set.csv")
PIPELINE_PKL = os.path.join(OUTPUT_DIR, "featurenet_3aux_pipeline.pkl")   # aux TMs + feature recipe + main TMs

# ---- Aux cache ------------------------------------------------------------------
# Aux TMs are the slow part and never see main labels, so they are trained once and
# saved here. Later runs (even after a kernel restart) load them and skip straight to
# the main TMs. The cache is retrained automatically when any aux-relevant setting or
# any input file changes (see the cache cell). Set AUX_CACHE = None to disable.
AUX_CACHE = os.path.join(OUTPUT_DIR, "featurenet_3aux_aux_cache_epo50.pkl")  
FORCE_RETRAIN_AUX = False   # True = ignore the cache once and retrain (then overwrite it)

assert len(MAIN_ISOFORM_COLS) == 4
assert (not USE_AUX_B) or len(AUX_B_TARGET_COLS) == len(MAIN_ISOFORM_COLS)
assert (not USE_AUX_C) or set(AUX_C_COL_MAP.values()) <= set(MAIN_ISOFORM_COLS), (
    f"AUX_C_COL_MAP values must be MAIN_ISOFORM_COLS entries: "
    f"{sorted(set(AUX_C_COL_MAP.values()) - set(MAIN_ISOFORM_COLS))} are not"
)
assert len(set(AUX_C_COL_MAP.values())) == len(AUX_C_COL_MAP), "two aux-C columns map to the same isoform"
assert AUX_A_FORMAT in {"long", "wide"}
assert AUX_FEATURE_MODE in {"clauses", "predictions", "both"}
assert AUX_FEATURE_SCOPE in {"matching", "all"}
assert OVERLAP_HANDLING in {"drop_from_aux", "oof", "ignore"}
assert isinstance(N_PARALLEL_JOBS, int) and N_PARALLEL_JOBS >= 1, "N_PARALLEL_JOBS must be an integer >= 1"
assert not (OVERLAP_HANDLING == "oof" and AUX_FEATURE_MODE != "predictions"), (
    "OVERLAP_HANDLING='oof' only works with AUX_FEATURE_MODE='predictions': each fold "
    "model learns different clauses, so clause outputs cannot be swapped between them. "
    "Use 'drop_from_aux' for clause features."
)
assert SPLIT_METHOD in {"scaffold", "kennard_stone", "random"}
assert CV_FOLDS == 0 or CV_FOLDS >= 2, "CV_FOLDS must be 0 (off) or >= 2"
assert set(AUX_A_ISOFORM_MAP.values()) <= set(MAIN_ISOFORM_COLS)
assert set(AUX_A_THRESHOLDS) <= set(AUX_A_VALUE_COLS), (
    f"AUX_A_THRESHOLDS keys must be aux-A value columns: "
    f"{sorted(set(AUX_A_THRESHOLDS) - set(AUX_A_VALUE_COLS))} are not"
)

### Aux A: long (pooled) vs wide (per CYP isoform) format

Both settings read the same long aux A file. `AUX_A_FORMAT` only decides how many aux A TMs are trained and what they see.

**Long (`"long"`): one TM per lab value, CYP isoform one-hot encoding as 4 extra input bits**

Pros
- **More data per model**: each TM sees every isoform's rows (up to ~4× the data of a per isoform model), so clauses for patterns shared across isoforms are more reliable.
- **Helps sparse isoforms**: an isoform with few aux A measurements borrows strength from the others instead of getting a weak model of its own.
- **Isoform differences are still learnable**: a clause is an "AND", so it can learn "isoform 2 AND substructure X", or drop the isoform bit when the pattern is shared.
- **Fewer models** (2 instead of up to 8): fewer feature blocks and faster to run.

Cons
- **Hurts if isoforms behave differently**: when their structure–activity patterns barely overlap, pooling mostly adds noise; per isoform models are then better.
- **Imbalance**: the isoform with the most rows dominates the shared clauses (mitigated by `AUX_A_BALANCE_ISOFORMS`, which oversamples rarer isoforms).
- **Target scale**: if a lab value's range differs by isoform, scale per isoform (`AUX_A_SCALE_PER_ISOFORM = True`); this discards absolute differences between isoforms.
- **Capacity**: one TM must cover 4 tasks, so it needs more clauses (`AUX_A_LONG_CLAUSE_MULT`) and is slower per epoch.
- **Feature time**: each molecule is run through the model 4 times (once per isoform bit), and with `AUX_FEATURE_SCOPE = "all"` the 4 resulting blocks are correlated.

**How to choose:** run the notebook with each setting and compare main validation RMSE / R², especially for the isoform with the fewest aux-A rows.

#### Canonical SMILES, overlap and leakage (out-of-fold reminder)

Same molecule can be written as different SMILES strings in different files. RDKit **canonical SMILES** have been used as the molecule key, so overlaps between aux A, aux B, aux C, main and test are found reliably. Unparsable SMILES are reported and left out of the overlap check.

If an aux TM was trained on a molecule that is also in main, its features for that molecule are **in-sample** (memorised, so this is unrealistically accurate). The main TM then learns to over trust aux features, and at test time, on molecules the aux TM never saw, those features are weaker than it expects. Validation scores are also
inflated when validation molecules were in aux training.

**Options (`OVERLAP_HANDLING`)**

- `"drop_from_aux"` (default): remove molecules found in main **or test** from aux training. Every main and test molecule then gets out-of-sample aux features, built the same way. The downside is this leads to less aux data.

- `"oof"`: keep the overlapping molecules for aux training and give them **out-of-fold** predictions (a fold model that did not see that molecule). Use this when the overlap is large and dropping molecules would throw away too much aux data. Only valid for prediction features, because each fold model learns different clauses.

- `"ignore"`: for comparison only.

> **Reminder:** if the overlap report shows a meaningful overlap and you need the
> aux data, use `OVERLAP_HANDLING = "oof"` (with `AUX_FEATURE_MODE = "predictions"`).
> Folds are grouped by canonical SMILES, so all isoform rows of one molecule are held
> out together. Also note the main TM's own CV does *not* need to retrain the aux TMs:
> they never see main labels.

**Other leakage guards in this notebook**
- Thermometer thresholds and clause selection are fitted on **aux** data only, never on main validation or test molecules.
- Duplicate molecules inside main are reported; scaffold splitting keeps them on the same side of a split.
- Main ∩ test overlap is reported separately; that is a train/test duplicate, not an aux issue.

#### Data loading

The five CSV files are read from `DATA_DIR` (set in the configuration cell). Main, aux B, aux C and test sets are *wide*, but aux A set is *long*. SMILES are canonicalised on loading and fingerprints are cached per canonical SMILES.

In [ ]:
_fp_cache = {}


def featurize_keys(keys):
    """ECFP bits for canonical SMILES keys, computed once per unique molecule."""
    keys = list(keys)
    missing = [k for k in dict.fromkeys(keys) if k not in _fp_cache]
    if missing:
        df = pd.DataFrame({"SMILES": missing})
        df["Mol"] = df["SMILES"].apply(mol_from_smiles)
        fps = np.asarray(
            gen_ecfp_arr(mol_df=df, mol_col="Mol", fp_radius=FP_RADIUS, fp_size=FP_SIZE)
        ).astype(np.uint32)
        for k, row in zip(missing, fps):
            _fp_cache[k] = row
    return np.vstack([_fp_cache[k] for k in keys]) if keys else np.zeros((0, FP_SIZE), np.uint32)


def add_canonical(df, smiles_col, label):
    """Add a 'canon' key column; unparseable rows keep their raw SMILES as key."""
    df = df.copy()
    canon = df[smiles_col].map(canonical_smiles)
    bad = canon.isna()
    if bad.any():
        print(f"  WARNING {label}: {int(bad.sum())} unparseable SMILES (kept, excluded from overlap checks)")
    df["canon"] = canon.fillna(df[smiles_col].astype(str))
    df["_parsed"] = ~bad
    return df


def load_auxA_long(csv_path):
    """Aux A (long) -> tidy long frame: canon, isoform (main column name), lab values.

    Isoform labels are mapped to MAIN_ISOFORM_COLS; duplicate (molecule, isoform) rows
    are averaged. Used by both AUX_A_FORMAT options.
    """
    df = pd.read_csv(csv_path)
    raw_labels = df[AUX_A_ISOFORM_COL].astype(str)
    df[AUX_A_ISOFORM_COL] = raw_labels.map(lambda v: AUX_A_ISOFORM_MAP.get(v, v))
    labels = set(df[AUX_A_ISOFORM_COL].unique())
    matched = labels & set(MAIN_ISOFORM_COLS)
    unmapped = sorted(labels - set(MAIN_ISOFORM_COLS))
    if not matched:
        raise ValueError(
            "aux A isoform labels do not match MAIN_ISOFORM_COLS even after mapping.\n"
            f"  aux A labels: {sorted(labels)}\n  MAIN_ISOFORM_COLS: {MAIN_ISOFORM_COLS}\n"
            "Fix: add the missing entries to AUX_A_ISOFORM_MAP."
        )
    if unmapped:
        print(f"  WARNING aux A: unmapped labels ignored: {unmapped}")
    df = df[df[AUX_A_ISOFORM_COL].isin(MAIN_ISOFORM_COLS)]
    df = add_canonical(df, AUX_A_SMILES_COL, "aux A")
    tidy = (df.groupby(["canon", AUX_A_ISOFORM_COL], as_index=False)[AUX_A_VALUE_COLS].mean()
              .rename(columns={AUX_A_ISOFORM_COL: "isoform"}))
    parsed = set(df.loc[df["_parsed"], "canon"])
    tidy["_parsed"] = tidy["canon"].isin(parsed)
    for v in AUX_A_VALUE_COLS:
        n_iso = tidy.dropna(subset=[v]).groupby("isoform").size().reindex(MAIN_ISOFORM_COLS).fillna(0)
        print(f"  aux A :: {v} -> per-isoform n={n_iso.astype(int).to_dict()}")
    return tidy


# File names inside DATA_DIR (relative paths, so the project runs from any location)
MAIN_CSV = os.path.join(DATA_DIR, "train_df_trim.csv")
AUX_A_CSV = os.path.join(DATA_DIR, "train_single_trim.csv")
AUX_B_CSV = os.path.join(DATA_DIR, "train_emax_trim.csv")
AUX_C_CSV = os.path.join(DATA_DIR, "train_surrogate_cyp.csv")
TEST_CSV = os.path.join(DATA_DIR, "test_df.csv")

_required = [MAIN_CSV, TEST_CSV] + ([AUX_A_CSV] if USE_AUX_A else []) + ([AUX_B_CSV] if USE_AUX_B else []) + ([AUX_C_CSV] if USE_AUX_C else [])
LOAD_DATA = all(os.path.exists(p) for p in _required)

if LOAD_DATA:
    main_df = add_canonical(pd.read_csv(MAIN_CSV), MAIN_SMILES_COL, "main")
    main_keys = main_df["canon"].to_numpy(dtype=object)
    X_main = featurize_keys(main_keys)
    Y_main = main_df[MAIN_ISOFORM_COLS].to_numpy(dtype=np.float64)
    n_dup = len(main_keys) - len(set(main_keys))
    if n_dup:
        print(f"  NOTE main: {n_dup} duplicate molecules after canonicalisation "
              "(kept; scaffold split keeps duplicates on the same side)")
    main_scaffolds = None
    if SPLIT_METHOD == "scaffold" or CV_FOLDS >= 2:
        main_scaffolds = np.array([murcko_scaffold(k) for k in main_keys], dtype=object)
        print(f"main: {len(set(main_scaffolds))} unique scaffolds across {len(main_scaffolds)} molecules")

    test_df = add_canonical(pd.read_csv(TEST_CSV), TEST_SMILES_COL, "test")
    test_keys = test_df["canon"].to_numpy(dtype=object)
    X_test = featurize_keys(test_keys)
    TEST_HAS_LABELS = all(col in test_df.columns for col in TEST_TARGET_COLS)
    Y_test = test_df[TEST_TARGET_COLS].to_numpy(dtype=np.float64) if TEST_HAS_LABELS else None

    auxA_long = load_auxA_long(AUX_A_CSV) if USE_AUX_A else None
    if USE_AUX_B:
        auxB_df = add_canonical(pd.read_csv(AUX_B_CSV), AUX_B_SMILES_COL, "aux B")
        auxB_df = auxB_df.groupby("canon", as_index=False).agg(
            {**{c: "mean" for c in AUX_B_TARGET_COLS}, "_parsed": "max"})
    else:
        auxB_df = None
    if USE_AUX_C:
        auxC_df = add_canonical(pd.read_csv(AUX_C_CSV), AUX_C_SMILES_COL, "aux C")
        missing_c = [c for c in AUX_C_COL_MAP if c not in auxC_df.columns]
        if missing_c:
            raise ValueError(f"aux C file has no column(s) {missing_c}; check AUX_C_COL_MAP keys")
        auxC_df = auxC_df.groupby("canon", as_index=False).agg(
            {**{c: "mean" for c in AUX_C_COL_MAP}, "_parsed": "max"})
        print("  aux C :: per-column n=" + str({f"{c}->{iso}": int(auxC_df[c].notna().sum())
                                               for c, iso in AUX_C_COL_MAP.items()}))
    else:
        auxC_df = None

    _test_mode = "labelled -> metrics" if TEST_HAS_LABELS else "unlabelled -> prediction only"
    print(f"main X:{X_main.shape} Y:{Y_main.shape} | test X:{X_test.shape} ({_test_mode})")
    print(f"USE_AUX_A={USE_AUX_A} (format={AUX_A_FORMAT}) | USE_AUX_B={USE_AUX_B} | USE_AUX_C={USE_AUX_C}")
else:
    _missing = [p for p in _required if not os.path.exists(p)]
    print(f"Data files not found: {_missing}. Check DATA_DIR and the file names "
          "(paths are relative to the notebook's folder).")

#### Overlap report (canonical SMILES)

This counts unique, parsable molecules shared between data files. Molecules in main and test sets that also appear in an aux data file are the ones `OVERLAP_HANDLING` deals with.

In [ ]:
if LOAD_DATA:
    def _parsed_keys(df):
        return set(df.loc[df["_parsed"], "canon"])

    sets = {"main": _parsed_keys(main_df), "test": _parsed_keys(test_df)}
    if USE_AUX_A:
        sets["aux A"] = _parsed_keys(auxA_long)
    if USE_AUX_B:
        sets["aux B"] = _parsed_keys(auxB_df)
    if USE_AUX_C:
        sets["aux C"] = _parsed_keys(auxC_df)

    names = list(sets)
    overlap_tbl = pd.DataFrame(
        [[len(sets[a] & sets[b]) for b in names] for a in names], index=names, columns=names
    )
    print("Unique-molecule overlap counts (diagonal = molecules in that file):")
    print(overlap_tbl)

    # Molecules whose aux features must be out-of-sample
    PROTECTED_KEYS = sets["main"] | sets["test"]
    for aux in [n for n in names if n.startswith("aux")]:
        shared = sets[aux] & PROTECTED_KEYS
        pct = 100 * len(shared) / max(len(sets[aux]), 1)
        print(f"{aux}: {len(shared)} of {len(sets[aux])} molecules ({pct:.1f}%) are also in main/test")
        if shared and OVERLAP_HANDLING == "ignore":
            print(f"  WARNING: OVERLAP_HANDLING='ignore' -> these get in-sample {aux} features (leaky).")
        elif shared and OVERLAP_HANDLING == "drop_from_aux" and pct > 30:
            print(f"  NOTE: dropping {pct:.0f}% of {aux}. If that is too much aux data to lose, "
                  "consider OVERLAP_HANDLING='oof' with AUX_FEATURE_MODE='predictions'.")
    if sets["main"] & sets["test"]:
        print(f"NOTE: {len(sets['main'] & sets['test'])} molecules appear in BOTH main and test "
              "(train/test duplicates, independent of the aux data).")

#### Auxiliary TMs to feature blocks

Each aux TM is trained once. A **feature block** is the set of bits one aux TM produces for one CYP isoform setting (clause outputs and/or thermometer bits). Feature blocks are tagged with their source (`A`, `B` or `C`) and CYP isoform, which is how `AUX_FEATURE_SCOPE` and the ablation study choose them.

In [ ]:
_rng = np.random.default_rng(RANDOM_SEED)


def iso_onehot(n, k):
    oh = np.zeros((n, len(MAIN_ISOFORM_COLS)), dtype=np.uint32)
    oh[:, k] = 1
    return oh


def balanced_rows(n, balance_labels=None):
    """Row order for one aux training: all rows, or with rarer groups oversampled.

    Random draws use the notebook's _rng, always in the main process and in the same
    order as a one-at-a-time run, so parallel training gives identical models.
    """
    idx = np.arange(n)
    if balance_labels is not None:
        groups = [np.flatnonzero(balance_labels == g) for g in np.unique(balance_labels)]
        target = max(len(g) for g in groups)
        idx = np.concatenate([
            g if len(g) == target else np.concatenate([g, _rng.choice(g, target - len(g))])
            for g in groups
        ])
    return idx


def _fit_aux_job(X_u8, y, rows, n_clauses, T_, s_, epochs, seed):
    """Worker: train one aux TMRegressor on rows of X (0/1 bits stored as uint8)."""
    import numpy as np
    from tmu.models.regression.vanilla_regressor import TMRegressor
    model = TMRegressor(number_of_clauses=n_clauses, T=T_, s=s_,
                        weighted_clauses=True, platform="CPU", seed=seed)
    Xb = np.ascontiguousarray(X_u8[rows], dtype=np.uint32)
    yb = y[rows]
    for _ in range(epochs):
        model.fit(Xb, yb)
    del Xb, yb
    # free tmu's internal copy of the training data before the model is sent back
    model.X_train = np.zeros(0, dtype=np.uint32)
    model.X_test = np.zeros(0, dtype=np.uint32)
    for attr in ("encoded_X_train", "encoded_X_test"):
        if hasattr(model, attr):
            setattr(model, attr, None)
    return model


def select_clauses(model, X_train, top_n):
    """Top-N informative clauses: weight x spread of firing (never/always firing = useless)."""
    out = model.transform(X_train).astype(np.float64)
    p = out.mean(axis=0)
    w = np.asarray(model.weight_bank.get_weights(), dtype=np.float64)
    score = np.abs(w) * np.sqrt(p * (1.0 - p))
    keep = np.argsort(score)[::-1][:top_n]
    return np.sort(keep[score[keep] > 0])


def slim_tm(model):
    """Drop tmu's cached copies of the last training/prediction inputs.

    tmu keeps a copy of the last X it was fit on and predicted on (plus an encoded
    version). They are only a speed cache, rebuilt on the next fit/predict, so clearing
    them never changes results. Done right after each aux TM trains and predicts (with
    out-of-fold handling there can be dozens of aux TMs, each otherwise holding its own
    copy in memory), and again before pickling.
    """
    model.X_train = np.zeros(0, dtype=np.uint32)
    model.X_test = np.zeros(0, dtype=np.uint32)
    for attr in ("encoded_X_train", "encoded_X_test"):
        if hasattr(model, attr):
            setattr(model, attr, None)
    return model


def fit_thermometer(values, n_bits):
    qs = np.quantile(values, np.linspace(0, 1, n_bits + 2)[1:-1])
    return np.unique(qs)


def thermometer(values, thresholds):
    return (np.asarray(values)[:, None] >= thresholds[None, :]).astype(np.uint32)


class AuxModel:
    """One aux TM plus its feature recipe (selected clauses, thermometer thresholds)
    and, in OOF mode, the fold models used for molecules it was trained on."""

    def __init__(self, name, source, X, y, keys, n_clauses, epochs,
                 long_iso_idx=None, isoform=None, bounds=None, cutoff=None):
        # bounds: scale01 (lo, hi) of the targets, to map predictions back to original
        #         units -- a tuple, or for long models a dict {isoform index: (lo, hi)}.
        # cutoff: known cutoff in original units -> one extra "pred < cutoff" bit
        #         (below the cutoff is the range of interest).
        self.name, self.source, self.isoform = name, source, isoform
        self.bounds, self.cutoff = bounds, cutoff
        self.is_long = long_iso_idx is not None
        keys = np.asarray(keys, dtype=object)

        # --- leakage handling -------------------------------------------------
        overlap = np.array([k in PROTECTED_KEYS for k in keys])
        if OVERLAP_HANDLING == "drop_from_aux" and overlap.any():
            keep = ~overlap
            print(f"    {name}: dropped {int(overlap.sum())} rows of main/test molecules")
            X, y, keys = X[keep], y[keep], keys[keep]
            long_iso_idx = long_iso_idx[keep] if self.is_long else None
        self.train_keys = set(keys)

        Xin = np.hstack([X, self._onehots(long_iso_idx)]) if self.is_long else X
        bal = long_iso_idx if (self.is_long and AUX_A_BALANCE_ISOFORMS) else None
        # Training is deferred: this only records the jobs (rows to train on) and makes
        # every random draw, in the same order as before. train_aux_jobs() runs them.
        self._X = np.ascontiguousarray(Xin, dtype=np.uint8)   # 0/1 bits; 4x smaller than uint32
        self._y = y
        self._n_clauses, self._epochs, self._n = n_clauses, epochs, len(y)
        self._jobs = [("full", balanced_rows(len(y), bal))]

        # --- out-of-fold models (prediction features only) ----------------------
        self.fold_of, self.fold_models = {}, []
        if OVERLAP_HANDLING == "oof" and (overlap.any()):
            uniq = np.array(sorted(set(keys)), dtype=object)
            fold_ids = _rng.permutation(len(uniq)) % AUX_OOF_FOLDS   # grouped by molecule
            self.fold_of = dict(zip(uniq, fold_ids))
            row_fold = np.array([self.fold_of[k] for k in keys])
            for f in range(AUX_OOF_FOLDS):
                tr_rows = np.flatnonzero(row_fold != f)
                sub = balanced_rows(len(tr_rows), bal[tr_rows] if bal is not None else None)
                self._jobs.append(("fold", tr_rows[sub]))
        self.model = None

    def _finish(self, trained):
        """Attach trained TMs (full model first, then folds) and build the feature recipe."""
        self.model, self.fold_models = trained[0], list(trained[1:])
        Xin = np.ascontiguousarray(self._X, dtype=np.uint32)
        print(f"    {self.name}: trained on n={self._n}")
        self.clauses = select_clauses(self.model, Xin, TOP_CLAUSES_PER_MODEL) \
            if AUX_FEATURE_MODE in {"clauses", "both"} else None
        self.thresholds = fit_thermometer(self.model.predict(Xin), N_THERMO_BITS) \
            if AUX_FEATURE_MODE in {"predictions", "both"} else None
        slim_tm(self.model)     # predict() above cached a copy of Xin; free it
        if self.fold_models:
            print(f"    {self.name}: {len(self.fold_models)} OOF fold models trained")
        del self._X, self._y, self._jobs, self._n_clauses, self._epochs, self._n


    @staticmethod
    def _onehots(iso_idx):
        oh = np.zeros((len(iso_idx), len(MAIN_ISOFORM_COLS)), dtype=np.uint32)
        oh[np.arange(len(iso_idx)), iso_idx] = 1
        return oh

    def _query(self, X_fp, iso_k):
        return np.hstack([X_fp, iso_onehot(len(X_fp), iso_k)]) if self.is_long else X_fp

    def predict(self, X_fp, keys, iso_k=None):
        Xq = np.ascontiguousarray(self._query(X_fp, iso_k))
        pred = self.model.predict(Xq)
        if self.fold_models:   # OOF: molecules seen in aux training use their fold model
            keys = np.asarray(keys, dtype=object)
            for f, fm in enumerate(self.fold_models):
                rows = np.flatnonzero([self.fold_of.get(k) == f for k in keys])
                if rows.size:
                    pred[rows] = fm.predict(np.ascontiguousarray(Xq[rows]))
        return pred

    def _bounds_for(self, iso_k):
        return self.bounds.get(iso_k, self.bounds["all"]) if isinstance(self.bounds, dict) else self.bounds

    def features(self, X_fp, keys, iso_k=None):
        parts, names = [], []
        tag = self.name + (f"|{MAIN_ISOFORM_COLS[iso_k]}" if self.is_long else "")
        if self.clauses is not None:
            out = self.model.transform(np.ascontiguousarray(self._query(X_fp, iso_k)))
            parts.append(out[:, self.clauses].astype(np.uint32))
            names += [f"{tag}::clause{c}" for c in self.clauses]
        pred = None
        if self.thresholds is not None:
            pred = self.predict(X_fp, keys, iso_k)
            parts.append(thermometer(pred, self.thresholds))
            names += [f"{tag}::pred>={t:.3g}" for t in self.thresholds]
        if self.cutoff is not None:
            if pred is None:
                pred = self.predict(X_fp, keys, iso_k)   # OOF-aware like the thermometer bits
            raw = unscale01(pred, self._bounds_for(iso_k))  # back to original units
            parts.append((raw < self.cutoff).astype(np.uint32)[:, None])   # 1 = below = of interest
            names.append(f"{tag}::pred<cutoff({self.cutoff:g})")
        return np.hstack(parts), names


def build_blocks(aux_models):
    """Feature blocks for main and test: one per (aux model, isoform setting)."""
    blocks = []
    for am in aux_models:
        settings = range(len(MAIN_ISOFORM_COLS)) if am.is_long else [None]
        for k in settings:
            Fm, names = am.features(X_main, main_keys, k)
            Ft, _ = am.features(X_test, test_keys, k)
            iso = MAIN_ISOFORM_COLS[k] if am.is_long else am.isoform
            blocks.append(dict(source=am.source, isoform=iso, model=am.name,
                               main=Fm, test=Ft, names=names))
        for m in [am.model] + am.fold_models:   # free the main/test copies predict() cached
            slim_tm(m)
    return blocks


def train_aux_jobs(aux_models):
    """Run every prepared aux training, up to N_PARALLEL_JOBS at a time, then finish
    each aux model in order. Biggest jobs start first so the cores stay busy."""
    jobs = [(i, j) for i, am in enumerate(aux_models) for j in range(len(am._jobs))]
    if not jobs:
        return
    cost = {(i, j): len(aux_models[i]._jobs[j][1]) * aux_models[i]._n_clauses * aux_models[i]._epochs
            for i, j in jobs}
    order = sorted(jobs, key=lambda ij: -cost[ij])
    n_workers = max(1, min(N_PARALLEL_JOBS, len(jobs), os.cpu_count() or 1))
    print(f"Training {len(jobs)} aux TMs ({len(aux_models)} aux models"
          f"{' incl. OOF folds' if OVERLAP_HANDLING == 'oof' else ''}) on {n_workers} parallel worker(s)")
    results = Parallel(n_jobs=n_workers, verbose=10 if n_workers > 1 else 0, mmap_mode="r")(
        delayed(_fit_aux_job)(
            aux_models[i]._X, aux_models[i]._y, aux_models[i]._jobs[j][1],
            aux_models[i]._n_clauses, int(AUX_T_PER_CLAUSE * aux_models[i]._n_clauses),
            AUX_S, aux_models[i]._epochs, RANDOM_SEED)
        for i, j in order)
    trained = dict(zip(order, results))
    for i, am in enumerate(aux_models):
        am._finish([trained[(i, j)] for j in range(len(am._jobs))])

#### Train or load the aux TMs

The **cache** holds the trained aux TMs and feature blocks they produce for main and test datasets. On a later run, the notebook checks a *fingerprint* of everything the aux step depends on: the aux settings, column names, ECFP settings, seed, and the **contents** of all four CSV files (main and test matter because they decide which overlapping molecules are dropped or held out). If it matches, the cache is loaded in seconds. If not, the mismatching items are listed and the aux TMs are retrained and the cache overwritten.

**Safe to change without aux retraining:** main TM settings (`N_CLAUSES`, `T`, `S`, `EPOCHS_MAIN`), the validation split, `CV_FOLDS`, `AUX_FEATURE_SCOPE`, Optuna, ablation study.

**Changes that trigger aux retraining:** anything in the aux sections of the configuration cell (`USE_AUX_A`, `USE_AUX_B` or `USE_AUX_C`) or any edit to a data CSV file. To force a retraining, set `FORCE_RETRAIN_AUX = True`.

**Parallel training:** with `N_PARALLEL_JOBS > 1`, the independent aux trainings (each aux model and its out-of-fold fold models) run side by side. It runs the biggest one first and joblib prints a `Done N tasks | elapsed …` line as they finish. The models, features and cache are identical to a one-at-a-time run, and `N_PARALLEL_JOBS` is not part of the cache check. Memory grows with the number of jobs (aux C's large training sets take ~1–1.5 GB per job), so check memory usage via `free -h` before raising it.

In [ ]:
def train_aux_models():
    """Train every enabled aux TM (the slow step)."""
    aux_models = []
    if LOAD_DATA and USE_AUX_A:
        print(f"Aux A ({AUX_A_FORMAT} format)")
        for v in AUX_A_VALUE_COLS:
            rows = auxA_long.dropna(subset=[v])
            cutoff = AUX_A_THRESHOLDS.get(v)
            if cutoff is not None:   # balance check: a nearly one-sided cutoff adds little
                frac = (rows[v] < cutoff).groupby(rows["isoform"]).mean().reindex(MAIN_ISOFORM_COLS)
                print(f"    {v}: share of rows below cutoff {cutoff:g} per isoform: "
                      + ", ".join(f"{i}={f:.0%}" for i, f in frac.items() if pd.notna(f)))
            if AUX_A_FORMAT == "long":
                iso_idx = rows["isoform"].map(MAIN_ISOFORM_COLS.index).to_numpy()
                y = rows[v].to_numpy(dtype=np.float64)
                ys, global_b = scale01(y)
                bounds = {"all": global_b}
                if AUX_A_SCALE_PER_ISOFORM:
                    for k in np.unique(iso_idx):
                        ys[iso_idx == k], bounds[int(k)] = scale01(y[iso_idx == k])
                aux_models.append(AuxModel(
                    f"A[{v}]", "A", featurize_keys(rows["canon"]), ys, rows["canon"].to_numpy(dtype=object),
                    AUX_N_CLAUSES * AUX_A_LONG_CLAUSE_MULT, EPOCHS_AUX_A, long_iso_idx=iso_idx,
                    bounds=bounds, cutoff=cutoff,
                ))
            else:   # wide: one TM per lab value x isoform
                for iso in MAIN_ISOFORM_COLS:
                    r = rows[rows["isoform"] == iso]
                    if len(r) < 10:
                        print(f"    A[{v}|{iso}]: only {len(r)} rows — skipped")
                        continue
                    ys, b = scale01(r[v].to_numpy(dtype=np.float64))
                    aux_models.append(AuxModel(
                        f"A[{v}|{iso}]", "A", featurize_keys(r["canon"]), ys,
                        r["canon"].to_numpy(dtype=object), AUX_N_CLAUSES, EPOCHS_AUX_A, isoform=iso,
                        bounds=b, cutoff=cutoff,
                    ))

    if LOAD_DATA and USE_AUX_B:
        print("Aux B")
        for iso, bcol in zip(MAIN_ISOFORM_COLS, AUX_B_TARGET_COLS):
            r = auxB_df.dropna(subset=[bcol])
            if len(r) < 10:
                print(f"    B[{iso}]: only {len(r)} rows — skipped")
                continue
            ys, _ = scale01(r[bcol].to_numpy(dtype=np.float64))
            aux_models.append(AuxModel(
                f"B[{iso}]", "B", featurize_keys(r["canon"]), ys,
                r["canon"].to_numpy(dtype=object), AUX_N_CLAUSES, EPOCHS_AUX_B, isoform=iso,
            ))

    if LOAD_DATA and USE_AUX_C:
        print("Aux C")
        for ccol, iso in AUX_C_COL_MAP.items():
            r = auxC_df.dropna(subset=[ccol])
            if len(r) < 10:
                print(f"    C[{iso}]: only {len(r)} rows — skipped")
                continue
            ys, _ = scale01(r[ccol].to_numpy(dtype=np.float64))
            aux_models.append(AuxModel(
                f"C[{iso}]", "C", featurize_keys(r["canon"]), ys,
                r["canon"].to_numpy(dtype=object), AUX_N_CLAUSES, EPOCHS_AUX_C, isoform=iso,
            ))
    train_aux_jobs(aux_models)   # the slow part: all trainings, N_PARALLEL_JOBS at a time
    return aux_models


def _file_md5(path):
    h = hashlib.md5()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def aux_fingerprint():
    """Everything the aux TMs and their feature blocks depend on."""
    return dict(
        cache_version=4,   # bump when the aux feature recipe changes
        files={os.path.basename(p): _file_md5(p) for p in _required},   # name + contents, not folder
        FP_RADIUS=FP_RADIUS, FP_SIZE=FP_SIZE, RANDOM_SEED=RANDOM_SEED,
        USE_AUX_A=USE_AUX_A, USE_AUX_B=USE_AUX_B, USE_AUX_C=USE_AUX_C,
        AUX_A_FORMAT=AUX_A_FORMAT, AUX_A_SCALE_PER_ISOFORM=AUX_A_SCALE_PER_ISOFORM,
        AUX_A_BALANCE_ISOFORMS=AUX_A_BALANCE_ISOFORMS,
        AUX_FEATURE_MODE=AUX_FEATURE_MODE, TOP_CLAUSES_PER_MODEL=TOP_CLAUSES_PER_MODEL,
        N_THERMO_BITS=N_THERMO_BITS, AUX_A_THRESHOLDS=dict(AUX_A_THRESHOLDS),
        OVERLAP_HANDLING=OVERLAP_HANDLING, AUX_OOF_FOLDS=AUX_OOF_FOLDS,
        AUX_N_CLAUSES=AUX_N_CLAUSES, AUX_T_PER_CLAUSE=AUX_T_PER_CLAUSE, AUX_S=AUX_S,
        AUX_A_LONG_CLAUSE_MULT=AUX_A_LONG_CLAUSE_MULT,
        EPOCHS_AUX_A=EPOCHS_AUX_A, EPOCHS_AUX_B=EPOCHS_AUX_B, EPOCHS_AUX_C=EPOCHS_AUX_C,
        MAIN_SMILES_COL=MAIN_SMILES_COL, MAIN_ISOFORM_COLS=list(MAIN_ISOFORM_COLS),
        TEST_SMILES_COL=TEST_SMILES_COL,
        AUX_A_SMILES_COL=AUX_A_SMILES_COL, AUX_A_ISOFORM_COL=AUX_A_ISOFORM_COL,
        AUX_A_VALUE_COLS=list(AUX_A_VALUE_COLS), AUX_A_ISOFORM_MAP=dict(AUX_A_ISOFORM_MAP),
        AUX_B_SMILES_COL=AUX_B_SMILES_COL, AUX_B_TARGET_COLS=list(AUX_B_TARGET_COLS),
        AUX_C_SMILES_COL=AUX_C_SMILES_COL, AUX_C_COL_MAP=dict(AUX_C_COL_MAP),
    )


def load_aux_cache(fp):
    """(aux_models, blocks) from AUX_CACHE if its fingerprint matches, else None."""
    if not AUX_CACHE or FORCE_RETRAIN_AUX or not os.path.exists(AUX_CACHE):
        if FORCE_RETRAIN_AUX:
            print("FORCE_RETRAIN_AUX=True -> retraining aux TMs")
        return None
    try:
        with open(AUX_CACHE, "rb") as fh:
            cached = pickle.load(fh)
    except Exception as exc:   # corrupt file, or a class/library changed since saving
        print(f"Aux cache unreadable ({type(exc).__name__}: {exc}) -> retraining")
        return None
    old = cached.get("fingerprint", {})
    changed = sorted(k for k in set(old) | set(fp) if old.get(k) != fp.get(k))
    if changed:
        if "files" in changed:
            changed.remove("files")
            changed += [f"file:{p}" for p in sorted(set(old.get("files", {})) | set(fp["files"]))
                        if old.get("files", {}).get(p) != fp["files"].get(p)]
        print(f"Aux cache is stale (changed: {', '.join(changed)}) -> retraining")
        return None
    return cached["aux_models"], cached["blocks"]


aux_models, blocks = [], []
if LOAD_DATA:
    _fp = aux_fingerprint()
    _cached = load_aux_cache(_fp)
    if _cached is not None:
        aux_models, blocks = _cached
        print(f"Loaded {len(aux_models)} aux TMs and {len(blocks)} feature blocks from {AUX_CACHE}")
    else:
        aux_models = train_aux_models()
        blocks = build_blocks(aux_models)
        if AUX_CACHE:
            for am in aux_models:
                for m in [am.model] + am.fold_models:
                    slim_tm(m)
            with open(AUX_CACHE, "wb") as fh:
                pickle.dump(dict(fingerprint=_fp, aux_models=aux_models, blocks=blocks), fh)
            print(f"Saved aux cache -> {AUX_CACHE} ({os.path.getsize(AUX_CACHE) / 1e6:.1f} MB)")

if blocks:
    print(pd.DataFrame([dict(model=b["model"], source=b["source"], isoform=b["isoform"],
                             n_bits=b["main"].shape[1]) for b in blocks]).to_string(index=False))

#### Assemble the augmented inputs

`ACTIVE_SOURCES` lists the aux data sources whose blocks are appended (the ablation cell overrides it). For CYP isoform `j`, `AUX_FEATURE_SCOPE` decides whether only isoform `j` blocks or all blocks are used.

In [ ]:
ACTIVE_SOURCES = [s for s, on in (("A", USE_AUX_A), ("B", USE_AUX_B), ("C", USE_AUX_C)) if on]


def assemble(j, sources=None):
    """(X_main_aug, X_test_aug, feature_names) for main isoform j."""
    sources = ACTIVE_SOURCES if sources is None else sources
    iso = MAIN_ISOFORM_COLS[j]
    use = [b for b in blocks if b["source"] in sources
           and (AUX_FEATURE_SCOPE == "all" or b["isoform"] == iso)]
    names = [f"ecfp{i}" for i in range(X_main.shape[1])] + [n for b in use for n in b["names"]]
    Xm = np.ascontiguousarray(np.hstack([X_main] + [b["main"] for b in use]).astype(np.uint32))
    Xt = np.ascontiguousarray(np.hstack([X_test] + [b["test"] for b in use]).astype(np.uint32))
    return Xm, Xt, names


if LOAD_DATA:
    for j, iso in enumerate(MAIN_ISOFORM_COLS):
        Xm, _, names = assemble(j)
        print(f"{iso}: {Xm.shape[1]} input bits ({Xm.shape[1] - X_main.shape[1]} from aux TMs)")

#### Hyperparameter search with Optuna (optional)

This is based on main TM model for one CYP isoform only in two stages where each can be used separately:

1. **Main only search** (`RUN_OPTUNA_MAIN_ONLY`): tunes `T` and `s` on ECFP inputs only. This is fast, leaves the aux cache untouched, and gives the like-for-like comparison with the baseline model e.g. RF.

2. **Confirmation search** (`RUN_OPTUNA_CONFIRM`): a short search on **main + aux** inputs, narrowed around the main only best (`T` within ×/÷ `CONFIRM_T_FACTOR`, `s` within ± `CONFIRM_S_DELTA`). If stage 1 hasn't run in this session, it centres on `CONFIRM_CENTER_T` / `CONFIRM_CENTER_S` (default: the config's `T` / `S`).

**Split:** on the tuning CYP isoform (`TUNE_ISOFORM_IDX`), set via using the isoform's index number in dataframe, always starting from the training cell's own split (`SPLIT_METHOD`: scaffold, Kennard-Stone or random).
- `OPTUNA_SPLIT = "inner"` (default): tuning uses a split **inside** that split's training part,
  so the validation molecules the notebook reports on are never used for tuning.
  `OPTUNA_INNER_METHOD` sets how the inner split is made:
  - `"scaffold"` (default): tunes for new scaffolds, the more conservative choice
  - `"kennard_stone"`: tunes for interpolation, matching a Kennard-Stone evaluation
- `OPTUNA_SPLIT = "validation"`: tunes directly on the training cell's validation molecules - validation scores may become optimistic with more tuning data.

**Epochs:** each trial scores validation RMSE (pIC50 units) after every epoch and keeps its **best epoch**; the trial's score is the RMSE at that epoch. So the search returns `T`, `s` **and** a recommended `EPOCHS_MAIN`. Weak trials are pruned early (median pruner).

**Run-time** (one CYP isoform): roughly trials × epochs × seconds per epoch, and halved by pruning. Rough estimate: at ~0.6 s/epoch, 50 trials × 100 epochs ≈ 25–50 min.

The suggested T, s & best epoch values can be entered into the configuration cell above to train the TM models further.

In [ ]:
RUN_OPTUNA_MAIN_ONLY = False
RUN_OPTUNA_CONFIRM = False
TUNE_ISOFORM_IDX = 0              # which main isoform to tune on (index into MAIN_ISOFORM_COLS)
OPTUNA_SPLIT = "inner"            # "inner" (keeps main validation clean) or "validation"
OPTUNA_INNER_FRACTION = 0.2       # inner split: share of the training part used for tuning validation
OPTUNA_INNER_METHOD = "scaffold"  # how the inner tuning split is made: "scaffold" or "kennard_stone"
OPTUNA_MAX_EPOCHS = 100           # epochs per trial (the best epoch within this is reported)

# Stage 1: main only search ranges
OPTUNA_N_TRIALS = 50
OPTUNA_T_PER_CLAUSE = (5, 100)    # T range as multiples of N_CLAUSES (searched on a log scale)
OPTUNA_S_RANGE = (1.0, 10.0)

# Stage 2: short confirmation on main + aux, narrowed around the stage 1 best
CONFIRM_N_TRIALS = 20
CONFIRM_T_FACTOR = 1.5            # T searched in [best_T / 1.5, best_T * 1.5]
CONFIRM_S_DELTA = 1.5             # s searched in [best_s - 1.5, best_s + 1.5] (min 1.0)
CONFIRM_CENTER_T = T              # used only if stage 1 hasn't run in this session
CONFIRM_CENTER_S = S

assert OPTUNA_SPLIT in {"inner", "validation"}
assert OPTUNA_INNER_METHOD in {"scaffold", "kennard_stone"}


def optuna_split(j):
    """Rows of isoform j for tuning -> (tune_train, tune_val, description).

    The outer split is the training cell's own split (main_split, following SPLIT_METHOD),
    so "inner" tuning never touches the validation molecules the notebook reports on.
    """
    global main_scaffolds
    if main_scaffolds is None:
        main_scaffolds = np.array([murcko_scaffold(k) for k in main_keys], dtype=object)
    idx = np.flatnonzero(np.isfinite(Y_main[:, j]))
    outer_tr, outer_va = main_split(idx)
    if OPTUNA_SPLIT == "validation" and len(outer_va) >= 2:
        return outer_tr, outer_va, f"main {SPLIT_METHOD} validation set"
    if OPTUNA_SPLIT == "validation":
        print("  VAL_FRACTION = 0 -> no main validation slice; using an inner split instead")
    if OPTUNA_INNER_METHOD == "kennard_stone":
        tr, va = kennard_stone_split(outer_tr, X_main, OPTUNA_INNER_FRACTION)
    else:
        tr, va = scaffold_split(outer_tr, main_scaffolds, OPTUNA_INNER_FRACTION)
    return tr, va, f"inner {OPTUNA_INNER_METHOD} split of the {SPLIT_METHOD} training part"


def run_search(label, X, j, n_trials, t_range, s_range, n_startup):
    """One Optuna study; returns (study, summary DataFrame of finished trials)."""
    tr, va, mode = optuna_split(j)
    y_tr_scaled, bounds = scale01(Y_main[tr, j])
    y_va = Y_main[va, j]
    X_tr = np.ascontiguousarray(X[tr])
    X_va = np.ascontiguousarray(X[va])
    print(f"\n[{label}] isoform {MAIN_ISOFORM_COLS[j]}: {X.shape[1]} input bits | "
          f"tune train={len(tr)}, tune val={len(va)} ({mode}) | "
          f"T in [{t_range[0]}, {t_range[1]}], s in [{s_range[0]:.2f}, {s_range[1]:.2f}]")

    def objective(trial):
        T_ = trial.suggest_int("T", int(t_range[0]), int(t_range[1]), log=True)
        s_ = trial.suggest_float("s", s_range[0], s_range[1])
        tm = TMRegressor(number_of_clauses=N_CLAUSES, T=T_, s=s_,
                         weighted_clauses=True, platform="CPU", seed=RANDOM_SEED)
        best_rmse, best_epoch, best_r2 = float("inf"), 0, float("nan")
        for epoch in range(1, OPTUNA_MAX_EPOCHS + 1):
            tm.fit(X_tr, y_tr_scaled)                     # one epoch per fit() call
            pred = unscale01(tm.predict(X_va), bounds)    # back to pIC50 units
            rmse = float(np.sqrt(mean_squared_error(y_va, pred)))
            if rmse < best_rmse:
                best_rmse, best_epoch = rmse, epoch
                best_r2 = float(r2_score(y_va, pred)) if len(y_va) >= 2 else float("nan")
            trial.set_user_attr("best_epoch", best_epoch)
            trial.set_user_attr("best_r2", best_r2)
            trial.report(rmse, epoch)
            if trial.should_prune():
                raise optuna.exceptions.TrialPruned()
        return best_rmse

    def progress(study, trial):
        if trial.state == optuna.trial.TrialState.COMPLETE:
            print(f"  trial {trial.number:3d}: T={trial.params['T']:>7d}  s={trial.params['s']:5.2f}  "
                  f"best val RMSE={trial.value:.4f} at epoch {trial.user_attrs['best_epoch']}")
        elif trial.state == optuna.trial.TrialState.PRUNED:
            print(f"  trial {trial.number:3d}: T={trial.params['T']:>7d}  s={trial.params['s']:5.2f}  pruned")

    optuna.logging.set_verbosity(optuna.logging.WARNING)
    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=n_startup, n_warmup_steps=25),
    )
    study.optimize(objective, n_trials=n_trials, n_jobs=1, callbacks=[progress])

    done = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
    summary = pd.DataFrame([dict(trial=t.number, T=t.params["T"], T_per_clause=t.params["T"] / N_CLAUSES,
                                 s=t.params["s"], best_epoch=t.user_attrs["best_epoch"],
                                 val_RMSE=t.value, val_R2=t.user_attrs["best_r2"]) for t in done])
    if summary.empty:
        print(f"[{label}] no trial completed (all pruned) — widen the ranges or lower n_warmup_steps")
        return study, summary
    summary = summary.sort_values("val_RMSE").reset_index(drop=True)
    b = summary.iloc[0]
    print(f"\n[{label}] BEST: T={int(b['T'])} ({b['T_per_clause']:.1f} x N_CLAUSES), s={b['s']:.3f}, "
          f"best epoch={int(b['best_epoch'])}, val RMSE={b['val_RMSE']:.4f}, val R2={b['val_R2']:.4f}")
    print(f"[{label}] top 5 of {len(summary)} completed trials "
          f"({len(study.trials) - len(summary)} pruned):")
    print(summary.head(5).round(4).to_string(index=False))
    print(f"[{label}] suggested config:  T = {int(b['T'])}   S = {b['s']:.3f}   EPOCHS_MAIN = {int(b['best_epoch'])}")
    return study, summary


import time
if LOAD_DATA and RUN_OPTUNA_MAIN_ONLY:
    t0 = time.time()
    X_tune, _, _ = assemble(TUNE_ISOFORM_IDX, [])           # ECFP only
    optuna_main_study, optuna_main_df = run_search(
        "main-only", X_tune, TUNE_ISOFORM_IDX, OPTUNA_N_TRIALS,
        (OPTUNA_T_PER_CLAUSE[0] * N_CLAUSES, OPTUNA_T_PER_CLAUSE[1] * N_CLAUSES),
        OPTUNA_S_RANGE, n_startup=10)
    print(f"[main-only] took {(time.time() - t0) / 60:.1f} min")

if LOAD_DATA and RUN_OPTUNA_CONFIRM:
    t0 = time.time()
    if "optuna_main_df" in globals() and not optuna_main_df.empty:
        cT, cS = int(optuna_main_df.iloc[0]["T"]), float(optuna_main_df.iloc[0]["s"])
        print(f"Confirmation centred on the main-only best: T={cT}, s={cS:.3f}")
    else:
        cT, cS = int(CONFIRM_CENTER_T), float(CONFIRM_CENTER_S)
        print(f"No main-only search in this session -> centred on CONFIRM_CENTER_T={cT}, CONFIRM_CENTER_S={cS:.3f}")
    X_tune, _, _ = assemble(TUNE_ISOFORM_IDX)                # ECFP + active aux sources
    optuna_confirm_study, optuna_confirm_df = run_search(
        "main + aux", X_tune, TUNE_ISOFORM_IDX, CONFIRM_N_TRIALS,
        (max(1, int(cT / CONFIRM_T_FACTOR)), int(cT * CONFIRM_T_FACTOR)),
        (max(1.0, cS - CONFIRM_S_DELTA), cS + CONFIRM_S_DELTA), n_startup=5)
    if "optuna_main_df" in globals() and not optuna_main_df.empty and not optuna_confirm_df.empty:
        d = optuna_confirm_df.iloc[0]["val_RMSE"] - optuna_main_df.iloc[0]["val_RMSE"]
        print(f"\nBest main + aux vs best main-only on the same tuning split: "
              f"val RMSE difference = {d:+.4f} (negative = aux helps)")
    print(f"[main + aux] took {(time.time() - t0) / 60:.1f} min")

#### Main TM per CYP isoform (trained once on the augmented input)

**Single split** (`SPLIT_METHOD`) is applied per CYP isoform to its labelled main molecules:
- `"scaffold"`: Largest scaffold groups fill the training set. Validation set will contain the rarest, unseen scaffolds. This is an extrapolation test (harder to reach optimal evaluation metrics).
- `"kennard_stone"`: Kennard-Stone split on Tanimoto distance of the ECFP bits. Training set spans the chemical space. Validation molecules sit *inside* this and are close to training molecules. This is an interpolation test (usually brings easier, more optimistic evaluation results), and analogues can be splitted across sides. This method builds a molecule × molecule distance matrix.
- `"random"`: Plain random split (`VAL_RANDOM_STATE`).

CV and the CV ablation keep scaffold-grouped folds for `"scaffold"` and `"kennard_stone"` (Kennard-Stone defines one split, not folds). The Optuna tuning split is always scaffold-based.

In [ ]:
def fit_epochs(model, X, y_scaled, n_epochs):
    for _ in range(n_epochs):
        model.fit(X, y_scaled)


def column_mask(Y, j):
    """Rows of column j that are finite (drop NaN/inf for that isoform)."""
    col = Y[:, j]
    return col, np.isfinite(col)


def make_split(idx):
    return main_split(idx)   # helper cell; same split the Optuna cell tunes within


def train_main_model(j, Xm, split=None, verbose=True):
    """Fresh main TMRegressor for isoform j on augmented inputs Xm."""
    col, m = column_mask(Y_main, j)
    if m.sum() == 0:
        print(f"    isoform[{j}] SKIPPED — no finite main targets")
        return None, None, None, None
    tr_idx, va_idx = split if split is not None else make_split(np.flatnonzero(m))
    model = TMRegressor(number_of_clauses=N_CLAUSES, T=T, s=S,
                        weighted_clauses=True, platform="CPU", seed=RANDOM_SEED)
    ys, bounds = scale01(col[tr_idx])
    fit_epochs(model, np.ascontiguousarray(Xm[tr_idx]), ys, EPOCHS_MAIN)
    if verbose:
        print(f"    isoform[{j}] main TM trained (train={len(tr_idx)}, val={len(va_idx)})")
    return model, bounds, tr_idx, va_idx


models, main_bounds_per_iso, split_per_iso, X_aug = {}, {}, {}, {}
if LOAD_DATA:
    for j, iso in enumerate(MAIN_ISOFORM_COLS):
        print(f"Training isoform {j} ({iso})")
        Xm, Xt, names = assemble(j)
        model_j, bounds_j, tr_j, va_j = train_main_model(j, Xm)
        if model_j is not None:
            models[iso], main_bounds_per_iso[iso] = model_j, bounds_j
            split_per_iso[iso] = (tr_j, va_j)
            X_aug[iso] = (Xm, Xt, names)

#### Scaffold-grouped cross-validation (optional)

This retrains only the main TM model per fold. The aux features stay fixed because the aux TMs never see main labels (molecule overlap is handled by `OVERLAP_HANDLING`). Folds are scaffold-grouped for `SPLIT_METHOD = "scaffold"` or `"kennard_stone"`, and per molecule for `"random"`.

In [ ]:
cv_df = None
if LOAD_DATA and CV_FOLDS >= 2:
    cv_rows = []
    for j, iso in enumerate(MAIN_ISOFORM_COLS):
        col, m = column_mask(Y_main, j)
        idx = np.flatnonzero(m)
        if len(idx) < 2 * CV_FOLDS:
            print(f"{iso}: only {len(idx)} labelled molecules — skipped")
            continue
        groups = main_scaffolds[idx] if SPLIT_METHOD != "random" else np.arange(len(idx))
        n_splits = min(CV_FOLDS, len(set(groups)))
        if n_splits < 2:
            print(f"{iso}: too few scaffold groups to cross-validate — skipped")
            continue
        Xm, _, _ = assemble(j)
        for fold, (tr_pos, va_pos) in enumerate(GroupKFold(n_splits=n_splits).split(idx, groups=groups)):
            tr_idx, va_idx = idx[tr_pos], idx[va_pos]
            model_f, bounds_f, _, _ = train_main_model(j, Xm, split=(tr_idx, va_idx), verbose=False)
            y_true = Y_main[va_idx, j]
            y_pred = unscale01(model_f.predict(Xm[va_idx]), bounds_f)
            cv_rows.append(dict(
                isoform=iso, fold=fold + 1, n_train=len(tr_idx), n_val=len(va_idx),
                val_R2=r2_score(y_true, y_pred) if len(va_idx) >= 2 else float("nan"),
                val_RMSE=np.sqrt(mean_squared_error(y_true, y_pred)),
                val_MAE=mean_absolute_error(y_true, y_pred),
            ))
            print(f"{iso} fold {fold + 1}/{n_splits} done")
    if cv_rows:
        cv_df = pd.DataFrame(cv_rows)
        print(cv_df.to_string(index=False))
        print("\nCross-validated summary (mean +/- std across folds):")
        print(cv_df.groupby("isoform")[["val_R2", "val_RMSE", "val_MAE"]].agg(["mean", "std"]).round(4))
elif LOAD_DATA:
    print("CV_FOLDS < 2 — cross-validation off; using the single split from the training cell.")

#### Ablation study (optional) - main only vs every combination of aux dataset A, B and C

The aux TM features are reused from the cache, and only the main TM model is retrained once for each study combination.

- `ABLATION_USE_CV = False`: one run per combination on the single validation split from the training cell (needs `VAL_FRACTION > 0`). This is quick but because this is on one fixed slice of data, small differences may be split noise.

- `ABLATION_USE_CV = True`: every combination is trained in **each** scaffold-grouped CV fold (`ABLATION_CV_FOLDS`) when using scaffolds as validation split, so all combinations are compared on identical folds. Mean ± std, the average **paired** difference from "main only", and in how many folds each "+" combination beats "main only" will be reported per combination. This works with any `VAL_FRACTION`. This will increase the notebook run-time (folds × combinations × isoforms main TM trainings (e.g. 5 × 8 × 4 = 160)).

**Combinations**: `ABLATION_ALL_SUBSETS` = `True` - every subset of the aux data sources (main only, +A, +B, +C, +A+B, +A+C, +B+C, +A+B+C = 8). If this is set to `False`, this will be a smaller combination as main only, each source alone, and all together (5 in total), leading to a reduction in run-time of about 40%.

**Reading the CV result**: a "+" combination may be convincing when its mean RMSE difference from main only is negative, clearly larger than its std, and it wins in most folds (e.g. 4 or 5 out of 5). If less folds are shown then this is likely due to noise.

In [ ]:
RUN_ABLATION = False
ABLATION_USE_CV = False   # True = compare combinations across scaffold-grouped CV folds
ABLATION_CV_FOLDS = 5
ABLATION_ALL_SUBSETS = False   # False = main only, each source alone, and all together


def ablation_combos():
    from itertools import combinations
    srcs = ACTIVE_SOURCES
    if ABLATION_ALL_SUBSETS:
        subsets = [list(c) for k in range(1, len(srcs) + 1) for c in combinations(srcs, k)]
    else:
        subsets = [[x] for x in srcs] + ([list(srcs)] if len(srcs) > 1 else [])
    return [("main only", [])] + [("main + " + " + ".join(c), c) for c in subsets]


def score_rows(y_true, y_pred):
    return dict(val_RMSE=np.sqrt(mean_squared_error(y_true, y_pred)),
                val_MAE=mean_absolute_error(y_true, y_pred),
                val_R2=r2_score(y_true, y_pred) if len(y_true) >= 2 else float("nan"))


def ablation_folds(j, n_folds):
    # Scaffold-grouped folds over isoform j's labelled rows (same scheme as the CV cell).
    col, m = column_mask(Y_main, j)
    idx = np.flatnonzero(m)
    groups = main_scaffolds[idx] if SPLIT_METHOD != "random" else np.arange(len(idx))
    n_splits = min(n_folds, len(set(groups)))
    if len(idx) < 2 * n_folds or n_splits < 2:
        return []
    return [(idx[tr], idx[va]) for tr, va in GroupKFold(n_splits=n_splits).split(idx, groups=groups)]


abl_df = None
if RUN_ABLATION and LOAD_DATA:
    combos = ablation_combos()
    abl = []
    if ABLATION_USE_CV:
        if SPLIT_METHOD != "random" and main_scaffolds is None:
            main_scaffolds = np.array([murcko_scaffold(k) for k in main_keys], dtype=object)
        for j, iso in enumerate(MAIN_ISOFORM_COLS):
            folds = ablation_folds(j, ABLATION_CV_FOLDS)
            if not folds:
                print(f"{iso}: too few labelled molecules / scaffold groups for CV ablation — skipped")
                continue
            X_by_combo = {label: assemble(j, srcs)[0] for label, srcs in combos}
            for f, (tr_idx, va_idx) in enumerate(folds, start=1):
                for label, _ in combos:
                    Xm = X_by_combo[label]
                    mdl, bnd, _, _ = train_main_model(j, Xm, split=(tr_idx, va_idx), verbose=False)
                    y_pred = unscale01(mdl.predict(Xm[va_idx]), bnd)
                    abl.append(dict(isoform=iso, fold=f, setup=label,
                                    **score_rows(Y_main[va_idx, j], y_pred)))
                print(f"{iso} fold {f}/{len(folds)} done ({len(combos)} combinations)")
    elif VAL_FRACTION > 0:
        for j, iso in enumerate(MAIN_ISOFORM_COLS):
            if iso not in split_per_iso:
                continue
            tr_idx, va_idx = split_per_iso[iso]
            if len(va_idx) < 2:
                continue
            for label, srcs in combos:
                Xm, _, _ = assemble(j, srcs)
                mdl, bnd, _, _ = train_main_model(j, Xm, split=(tr_idx, va_idx), verbose=False)
                y_pred = unscale01(mdl.predict(Xm[va_idx]), bnd)
                abl.append(dict(isoform=iso, fold=0, setup=label,
                                **score_rows(Y_main[va_idx, j], y_pred)))
                print(f"{iso} :: {label} done")
    else:
        print("Single-split ablation needs VAL_FRACTION > 0 (or set ABLATION_USE_CV = True).")

    if abl:
        abl_df = pd.DataFrame(abl)
        order = [label for label, _ in combos]
        if not ABLATION_USE_CV:
            print("\nValidation RMSE (single split; lower is better):")
            print(abl_df.pivot(index="isoform", columns="setup", values="val_RMSE")[order].round(4))
            print("\nValidation R2 (single split):")
            print(abl_df.pivot(index="isoform", columns="setup", values="val_R2")[order].round(4))
        else:
            summ = abl_df.groupby(["isoform", "setup"])[["val_RMSE", "val_R2"]].agg(["mean", "std"])
            rmse_txt = (summ[("val_RMSE", "mean")].map("{:.4f}".format) + " ± "
                        + summ[("val_RMSE", "std")].map("{:.4f}".format)).unstack()[order]
            print("\nCV validation RMSE, mean ± std across folds (lower is better):")
            print(rmse_txt.to_string())
            print("\nCV validation R2, mean across folds:")
            print(summ[("val_R2", "mean")].unstack()[order].round(4).to_string())

            # Paired comparison with "main only" on identical folds
            base = abl_df[abl_df["setup"] == "main only"].set_index(["isoform", "fold"])["val_RMSE"]
            paired = []
            for label in order[1:]:
                other = abl_df[abl_df["setup"] == label].set_index(["isoform", "fold"])["val_RMSE"]
                diff = (other - base).dropna()
                for iso, d in diff.groupby(level="isoform"):
                    paired.append(dict(isoform=iso, setup=label,
                                       mean_dRMSE=d.mean(), std_dRMSE=d.std(),
                                       folds_better=f"{int((d < 0).sum())}/{len(d)}"))
            if paired:
                pdf = pd.DataFrame(paired)
                print("\nPaired difference vs main only (RMSE(+aux) − RMSE(main only); negative = aux helps):")
                print(pdf.pivot(index="isoform", columns="setup", values="mean_dRMSE")[order[1:]].round(4).to_string())
                print("\nFolds where the combination beats main only:")
                print(pdf.pivot(index="isoform", columns="setup", values="folds_better")[order[1:]].to_string())

In [ ]:
# ---- Evaluation for each CYP isoform  + test set prediction ---------------------------
results = []
preds_cache = {}
NAN = float("nan")

if LOAD_DATA:
    id_cols = [c for c in [TEST_NAME_COL, TEST_SMILES_COL] if c in test_df.columns]
    test_pred_df = test_df[id_cols].copy()

    for j, iso in enumerate(MAIN_ISOFORM_COLS):
        if iso not in models:
            continue
        model = models[iso]
        bounds = main_bounds_per_iso[iso]
        Xm, Xt, _ = X_aug[iso]

        tr_idx, va_idx = split_per_iso[iso]
        y_main_true = Y_main[tr_idx, j]
        y_main_pred = unscale01(model.predict(Xm[tr_idx]), bounds)
        y_main_clf = (y_main_true >= ACTIVE_CUTOFF).astype(int)
        train_r2 = r2_score(y_main_true, y_main_pred)
        train_rmse = np.sqrt(mean_squared_error(y_main_true, y_main_pred))
        train_mae = mean_absolute_error(y_main_true, y_main_pred)

        val_r2 = val_rmse = val_mae = NAN
        y_val_true = y_val_pred = None
        if len(va_idx) >= 2:
            y_val_true = Y_main[va_idx, j]
            y_val_pred = unscale01(model.predict(Xm[va_idx]), bounds)
            val_r2 = r2_score(y_val_true, y_val_pred)
            val_rmse = np.sqrt(mean_squared_error(y_val_true, y_val_pred))
            val_mae = mean_absolute_error(y_val_true, y_val_pred)

        thr = select_best_threshold(y_main_clf, y_main_pred, MIN_PRECISION)

        y_test_pred = unscale01(model.predict(Xt), bounds)
        test_pred_df[f"pred_pIC50_{iso}"] = y_test_pred
        test_pred_df[f"pred_active_{iso}"] = (y_test_pred >= thr).astype(int)

        row = dict(
            isoform=iso, threshold=thr,
            train_R2=train_r2, train_RMSE=train_rmse, train_MAE=train_mae,
            val_R2=val_r2, val_RMSE=val_rmse, val_MAE=val_mae,
            test_RMSE=NAN, test_MAE=NAN, test_ROC_AUC=NAN, test_PRC_AUC=NAN,
            PPV=NAN, NPV=NAN, test_precision=NAN, test_recall=NAN,
        )
        y_test_clf, y_test_scores = None, None
        if TEST_HAS_LABELS:
            col_t, m_t = column_mask(Y_test, j)
            if m_t.sum() > 0:
                y_test_true = col_t[m_t]
                y_test_scores = y_test_pred[m_t]
                y_test_clf = (y_test_true >= ACTIVE_CUTOFF).astype(int)
                y_test_bin = (y_test_scores >= thr).astype(int)
                row["test_RMSE"] = np.sqrt(mean_squared_error(y_test_true, y_test_scores))
                row["test_MAE"] = mean_absolute_error(y_test_true, y_test_scores)
                if len(np.unique(y_test_clf)) > 1:
                    row["test_ROC_AUC"] = roc_auc_score(y_test_clf, y_test_scores)
                    row["test_PRC_AUC"] = prc_auc_score(y_test_clf, y_test_scores)
                    row["PPV"], row["NPV"] = ppv_npv_score(y_test_clf, y_test_scores)
                row["test_precision"] = precision_score(y_test_clf, y_test_bin, zero_division=0)
                row["test_recall"] = recall_score(y_test_clf, y_test_bin, zero_division=0)

        results.append(row)
        preds_cache[iso] = dict(
            y_main_true=y_main_true, y_main_pred=y_main_pred, thr=thr,
            y_test_clf=y_test_clf, y_test_scores=y_test_scores,
            train_R2=train_r2, train_RMSE=train_rmse, train_MAE=train_mae,
            y_val_true=y_val_true, y_val_pred=y_val_pred,
            val_R2=val_r2, val_RMSE=val_rmse, val_MAE=val_mae,
            test_precision=row["test_precision"], test_recall=row["test_recall"],
            test_PRC_AUC=row["test_PRC_AUC"],
        )

    results_df = pd.DataFrame(results).set_index("isoform")
    if not TEST_HAS_LABELS:
        results_df = results_df.dropna(axis=1, how="all")
    pd.set_option("display.float_format", lambda v: f"{v:.4f}")
    print(results_df)

    if VAL_FRACTION > 0 and "val_R2" in results_df.columns:
        gap = pd.DataFrame({
            "train_R2": results_df["train_R2"], "val_R2": results_df["val_R2"],
            "R2_gap": results_df["train_R2"] - results_df["val_R2"],
            "val_RMSE": results_df["val_RMSE"], "val_MAE": results_df["val_MAE"],
        })
        print("\nGeneralisation check (validation = held-out main data):")
        print(gap)

    test_pred_df.to_csv(PREDICTIONS_CSV, index=False)
    print(f"\nSaved {len(test_pred_df)} test predictions -> {PREDICTIONS_CSV}")
    print(test_pred_df.head())

In [ ]:
# ---- Which inputs does each main TM actually use? ---------------------------
# Counts how many main clauses include each feature (as x or NOT x), summed per
# source. A source the main TM barely uses is contributing little.
if LOAD_DATA:
    usage_rows = []
    for iso, model in models.items():
        _, _, names = X_aug[iso]
        n_feat = len(names)
        freq = np.asarray(model.literal_clause_frequency(), dtype=np.int64)
        per_feat = freq[:n_feat] + freq[n_feat:2 * n_feat]
        src = np.array(["ECFP" if n.startswith("ecfp") else n.split("::")[0] for n in names])
        for s in dict.fromkeys(src):
            sel = src == s
            usage_rows.append(dict(isoform=iso, input=s, n_bits=int(sel.sum()),
                                   clause_inclusions=int(per_feat[sel].sum()),
                                   per_bit=float(per_feat[sel].mean())))
    usage_df = pd.DataFrame(usage_rows)
    print(usage_df.to_string(index=False))

In [ ]:
# ---- Plots: one scatter plot per isoform, plus combined ROC and PR -------------
if LOAD_DATA:
    fig, axes = plt.subplots(2, 2, figsize=(13, 11), constrained_layout=True)
    fig.suptitle("Feature net: main-set true vs predicted per isoform — train vs held-out validation",
                 fontsize=14, fontweight="bold")
    for ax, iso in zip(axes.ravel(), list(preds_cache)):
        d = preds_cache[iso]
        ax.scatter(d["y_main_true"], d["y_main_pred"], alpha=0.45, s=18, label="Train")
        vals = [d["y_main_true"], d["y_main_pred"]]
        if d["y_val_true"] is not None:
            ax.scatter(d["y_val_true"], d["y_val_pred"], alpha=0.85, s=24,
                       color="tab:orange", edgecolor="none", label="Validation (held out)")
            vals += [d["y_val_true"], d["y_val_pred"]]
        ax.axhline(d["thr"], linestyle="--", color="tab:red", label=f"Threshold = {d['thr']:.2f}")
        ax.axvline(ACTIVE_CUTOFF, linestyle=":", color="tab:green", label=f"Active cut-off = {ACTIVE_CUTOFF:.2f}")
        lo = min(float(np.min(v)) for v in vals)
        hi = max(float(np.max(v)) for v in vals)
        ax.plot([lo, hi], [lo, hi], color="0.6", lw=1)
        ax.set(xlabel="True pIC50", ylabel="Predicted", title=iso, xlim=(lo, hi), ylim=(lo, hi))
        ax.text(
            0.02, 0.98,
            f"Train R2={d['train_R2']:.3f}  RMSE={d['train_RMSE']:.3f}  MAE={d['train_MAE']:.3f}"
            + (f"\nVal   R2={d['val_R2']:.3f}  RMSE={d['val_RMSE']:.3f}  MAE={d['val_MAE']:.3f}"
               if d["y_val_true"] is not None else "")
            + (f"\nTest P={d['test_precision']:.3f}  R={d['test_recall']:.3f}" if TEST_HAS_LABELS else ""),
            fontsize=9, ha="left", va="top", transform=ax.transAxes,
            bbox=dict(facecolor="white", edgecolor="tab:red", boxstyle="round,pad=0.3"),
        )
        ax.legend(loc="lower right", fontsize=8)
    plt.show()

if LOAD_DATA and TEST_HAS_LABELS:
    fig, (axr, axp) = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)
    for iso in preds_cache:
        d = preds_cache[iso]
        if d["y_test_clf"] is None or len(np.unique(d["y_test_clf"])) < 2:
            continue
        fpr, tpr, _ = roc_curve(d["y_test_clf"], d["y_test_scores"])
        axr.plot(fpr, tpr, lw=2, label=f"{iso} (AUC={auc(fpr, tpr):.3f})")
        pr, rc, _ = precision_recall_curve(d["y_test_clf"], d["y_test_scores"])
        axp.plot(rc, pr, lw=2, label=f"{iso} (PRC-AUC={d['test_PRC_AUC']:.3f})")
    axr.plot([0, 1], [0, 1], "--", color="0.5")
    axr.set(xlim=(0, 1), ylim=(0, 1.02), xlabel="False-positive rate", ylabel="True-positive rate", title="Test ROC")
    axr.legend(loc="lower right", fontsize=8, frameon=False)
    axp.set(xlim=(0, 1), ylim=(0, 1.02), xlabel="Recall", ylabel="Precision", title="Test precision-recall")
    axp.legend(loc="lower left", fontsize=8, frameon=False)
    plt.show()

#### Save the pipeline (optional) 

In [ ]:
SAVE_PIPELINE = False
if SAVE_PIPELINE and LOAD_DATA:
    for am in aux_models:
        for m in [am.model] + am.fold_models:
            slim_tm(m)
    for m in models.values():
        slim_tm(m)
    with open(PIPELINE_PKL, "wb") as fh:
        pickle.dump(dict(
            config=dict(FP_RADIUS=FP_RADIUS, FP_SIZE=FP_SIZE, AUX_A_FORMAT=AUX_A_FORMAT,
                        AUX_FEATURE_MODE=AUX_FEATURE_MODE, AUX_FEATURE_SCOPE=AUX_FEATURE_SCOPE,
                        ACTIVE_SOURCES=ACTIVE_SOURCES, MAIN_ISOFORM_COLS=MAIN_ISOFORM_COLS),
            aux_models=aux_models, main_models=models, main_bounds=main_bounds_per_iso,
        ), fh)
    print(f"Saved pipeline -> {PIPELINE_PKL}")